In [4]:
import cv2
import numpy as np
import tensorflow as tf
import glob
import os

model = tf.keras.models.load_model("./best_model.h5", compile=False)

In [5]:
CLASS_COLORS = {
    1: [255,   0,   0],   # ship -> red
    2: [128, 128, 128],   # sargassum -> gray
    3: [255, 255, 255],   # oil -> white
}

# Open video
cap = cv2.VideoCapture("../video/ManchaSat.mp4")

def preprocess_frame(frame):
    frame = cv2.resize(frame, (256, 256))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = gray[..., np.newaxis]
    return np.expand_dims(gray.astype("float32") / 255.0, axis=0)

def decode_mask(pred, frame_shape):
    mask = np.argmax(pred[0], axis=-1).astype(np.uint8)
    mask = cv2.resize(mask, (frame_shape[1], frame_shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask

def colorize_mask(mask):
    # Create a 3-channel RGB canvas
    mask_rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)

    # Ships = class 1 -> red
    mask_rgb[mask == 1] = (0, 0, 255)

    # Oil = class 6 -> white
    mask_rgb[mask == 3] = (255, 255, 255)

    # Sargassum = class 7 -> orange (BGR)
    mask_rgb[mask == 2] = (0, 165, 255)

    return mask_rgb

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    input_tensor = preprocess_frame(frame)
    pred = model.predict(input_tensor, verbose=0)

    mask = decode_mask(pred, frame.shape)
    mask_colored = colorize_mask(mask)

    oil_pixels = np.sum(mask == 3)
    total_pixels = mask.size

    cv2.putText(
        mask_colored,
        f"Area: {(oil_pixels/total_pixels)*100:.2f}%",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        1,
        cv2.LINE_AA,
    )

    # Show only the mask (not the original frame)
    cv2.imshow("Segmentation Mask", mask_colored)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


In [6]:
image_folder = "../grayscale_frames"
mask_folder = "../mask_frames"

# Load files
image_files = sorted(glob.glob(os.path.join(image_folder, "*.png")))
mask_files = sorted(glob.glob(os.path.join(mask_folder, "*.png")))

def preprocess_frame(frame):
    frame = cv2.resize(frame, (256, 256))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = gray[..., np.newaxis]
    return np.expand_dims(gray.astype("float32") / 255.0, axis=0)

def decode_mask(pred, frame_shape):
    mask = np.argmax(pred[0], axis=-1).astype(np.uint8)
    mask = cv2.resize(mask, (frame_shape[1], frame_shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask

# Counters
pred_total = 0
gt_total = 0
iou_total = 0
dice_total = 0
num_images = min(len(image_files), len(mask_files))

for img_path, mask_path in zip(image_files, mask_files):
    # Load input image and predict
    frame = cv2.imread(img_path)
    input_tensor = preprocess_frame(frame)
    pred = model.predict(input_tensor, verbose=0)
    pred_mask = decode_mask(pred, frame.shape)

    # Predicted oil mask (class 3)
    pred_oil = (pred_mask == 3).astype(np.uint8)

    # Ground-truth oil mask (assuming white=oil)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = cv2.resize(gt_mask, (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST)
    gt_oil = (gt_mask > 127).astype(np.uint8)

    # Pixel counts
    pred_total += np.sum(pred_oil)
    gt_total += np.sum(gt_oil)

    # Intersection and Union
    intersection = np.sum((pred_oil & gt_oil) == 1)
    union = np.sum((pred_oil | gt_oil) == 1)

    # IoU
    iou = intersection / union if union > 0 else 0
    iou_total += iou

    # Dice score = 2 * intersection / (pred + gt)
    denom = np.sum(pred_oil) + np.sum(gt_oil)
    dice = (2 * intersection / denom) if denom > 0 else 0
    dice_total += dice

# Averages
avg_pred = pred_total / num_images
avg_gt = gt_total / num_images
avg_iou = iou_total / num_images
avg_dice = dice_total / num_images

percent_diff = abs(avg_pred - avg_gt) / avg_gt * 100 if avg_gt > 0 else 0

print(f"Average predicted oil pixels: {avg_pred:.2f}")
print(f"Average ground-truth oil pixels: {avg_gt:.2f}")
print(f"Difference: {avg_pred - avg_gt:.2f} px")
print(f"Percent difference: {percent_diff:.2f}%")
print(f"Average IoU: {avg_iou:.3f}")
print(f"Average Dice: {avg_dice:.3f}")

Average predicted oil pixels: 315.51
Average ground-truth oil pixels: 315.12
Difference: 0.38 px
Percent difference: 0.12%
Average IoU: 0.929
Average Dice: 0.937
